# Pre-ADMET Filtering

This notebook filters the generated library based on structural alerts and physicochemical properties specialized for Intravenous (IV) administration.
It relaxes some Lipinski constraints suitable for IV drugs (e.g., TPSA, MW) while ensuring basic drug-likeness.

In [4]:
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem, FilterCatalog, QED, Descriptors, Lipinski
from rdkit.Chem.FilterCatalog import FilterCatalog, FilterCatalogParams
from IPython.display import SVG, display
from pathlib import Path
from loguru import logger
import numpy as np
from tqdm.auto import tqdm
from datetime import datetime
from itertools import product
from pprint import pprint
import io
import time
import sys 
import os

## Initialize Filter Catalog

Initialize RDKit's FilterCatalog with PAINS (Pan Assay Interference Compounds) structure alerts.

In [5]:
filters_params = FilterCatalogParams()
filters_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalog = FilterCatalog(filters_params)

logger.info("PAINS Filter Catalog Initialized.")

## Define Filtering Logic (IV Mode)

We define a custom filtering function `check_filters_iv_sepsis` tailored for IV administration:

1.  **PAINS/Brenk**: Disabled for this specific project because the Curcumin scaffold itself contains a diketone/enone group that triggers PAINS alerts. Enabling it would filter out the entire library.
2.  **TPSA (Topological Polar Surface Area)**:
    *   Typical Oral Limit: 140.
    *   IV Limit: relaxed. We need high polarity for solubility. We set a technical cap of 400 to remove extreme outliers.
3.  **Rotatable Bonds**:
    *   Allowed slightly more flexibility for IV (Limit > 12).
4.  **Molecular Weight (MW)**:
    *   Range: 150 - 700. IV antibiotics can be large (e.g., Vancomycin ~1400), but we keep it reasonable for small molecules.
5.  **QED (Quantitative Estimate of Drug-likeness)**:
    *   QED is biased towards oral drugs. We lower the threshold to 0.3 to retain good IV candidates.

In [6]:
def check_filters_iv_sepsis(mol):
    if mol is None:
        return False, "Invalid SMILES"

    # --- A. Structural Alerts (PAINS & Brenk) ---
    # Disabled for Curcumin scaffold (contains diketone/enone).
    # if catalog.HasMatch(mol):
    #     return False, "PAINS/Brenk Alert"

    # --- B. Polarity and Flexibility (IV Adapted) ---
    
    # TPSA
    tpsa = Descriptors.TPSA(mol)
    if tpsa > 400: 
        return False, "TPSA > 400 (Too Polar)"
    
    # Rotatable Bonds
    rot_bonds = Lipinski.NumRotatableBonds(mol)
    if rot_bonds > 12:
        return False, "RotBonds > 12"

    # --- C. Molecular Size ---
    mw = Descriptors.MolWt(mol)
    # Range 150-700 for small molecule IV candidates
    if mw < 150 or mw > 700: 
         return False, "MW Out of Range (150-700)"

    # --- D. QED (Drug-likeness) ---
    # Lowered threshold for IV
    try:
        qed_score = QED.qed(mol)
        if qed_score < 0.3:
            return False, f"Low QED (< 0.3)"
    except:
        return False, "QED Error"

    return True, "OK"

## Execution

Filter `ligands_ro5_filtered.smi` and save passing molecules to `ligands_pre_admet.smi`.

In [7]:
if input("Are you sure? [y/n] ").lower() == "y":
    input_filename = Path("..") / "data" / "interim" / "ligands_ro5_filtered.smi" 
    output_filename = Path("..") / "data" / "interim" / "ligands_pre_admet.smi"

    stats = {
        "Total": 0,
        "Passed": 0,
        "Invalid SMILES": 0,
        "PAINS/Brenk Alert": 0,
        "TPSA > 400 (Too Polar)": 0,
        "TPSA < 50 (Too Lipophilic)": 0,
        "RotBonds > 12": 0,
        "MW Out of Range (150-700)": 0,
        "Low QED (< 0.3)": 0,
        "QED Error": 0
    }

    logger.info(f"Starting filtering (IV Mode): {input_filename} -> {output_filename}")
    BATCH_SIZE = 10000 

    buffer = io.StringIO()
    counter_written = 0

    with open(input_filename, "r") as f_in, open(output_filename, "w") as f_out:
        lines = f_in.readlines()
        stats["Total"] = len(lines)
        
        for line in tqdm(lines, desc="Processing Molecules IV"):
            parts = line.strip().split()
            if len(parts) < 2: continue
            comb, smiles = parts[0], parts[1]
            
            mol = Chem.MolFromSmiles(smiles)
            
            is_pass, reason = check_filters_iv_sepsis(mol)

            if is_pass:
                counter_written += 1
                buffer.write(f"""{comb} {smiles}\n""")
                stats["Passed"] += 1
                
                if counter_written % BATCH_SIZE == 0:
                    f_out.write(buffer.getvalue())
                    buffer.seek(0)
                    buffer.truncate(0)
            else:
                stats[reason] = stats.get(reason, 0) + 1

        f_out.write(buffer.getvalue())

    logger.success("Filtering completed!")
    logger.info(f"Results saved to: {output_filename}")
    logger.info("-" * 30)
    logger.info(f"Total processed: {stats['Total']}")
    if stats['Total'] > 0:
        logger.info(f"Passed:  {stats['Passed']} ({(stats['Passed']/stats['Total'])*100:.1f}%)")
    logger.info("-" * 30)
    logger.info("Rejection reasons:")
    for reason, count in stats.items():
        if reason not in ["Total", "Passed"] and count > 0:
            logger.info(f"{reason}: {count}")
else:
    print("Aborted by user")